<a href="https://colab.research.google.com/github/wasit7/papapipeline/blob/main/Data_Quality_Engine_v1_3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# -*- coding: utf-8 -*-
"""
Data Quality Dashboard Engine - Version 1.3
Includes Data Engineer (Hive Partitioning) and QA Engineer (Interactive Visualization) features.
Updated to partition solely by run_date and simulates 5 days of data processing.
"""

import os
import sys
import datetime
import json
import pandas as pd
import numpy as np
import pyarrow as pa
import pyarrow.parquet as pq

try:
    import altair as alt
except ImportError:
    print("Warning: Altair not found. Visualizations will be disabled.")
    alt = None

# --- 1. SETTINGS & ENVIRONMENT PREP ---
try:
    from pandarallel import pandarallel
    pandarallel.initialize(progress_bar=True, verbose=0)
    HAS_PARALLEL = True
except ImportError:
    HAS_PARALLEL = False

# Detect if we are in an interactive IPython/Jupyter environment
def in_notebook():
    try:
        from IPython import get_ipython
        if 'IPKernelApp' not in get_ipython().config:  # pragma: no cover
            return False
    except ImportError:
        return False
    except AttributeError:
        return False
    return True

if in_notebook() and alt is not None:
    alt.renderers.enable('default')

TYPE_SPECS = {
    "string":   {"default": pd.NA,  "dtype": pa.string(),           "coerce": lambda x: x.astype("string")},
    "float":    {"default": np.nan, "dtype": pa.float64(),          "coerce": lambda x: pd.to_numeric(x, errors='coerce')},
    "datetime": {"default": pd.NaT, "dtype": pa.timestamp('ns'),    "coerce": lambda x: pd.to_datetime(x, errors='coerce')}
}

# --- 2. CORE ENGINE ---
class Monad:
    __slots__ = ['value', 'input_row', 'output_column', 'status', 'final_value', 'logs', 'stopped', 'step']
    def __init__(self, value, row, col):
        self.value = value; self.input_row = row; self.output_column = col
        self.status = 'pending'; self.final_value = None; self.logs = []; self.stopped = False; self.step = 0

    def _log(self, func, status, details=None):
        role = getattr(func, '_role', 'validator')
        name = getattr(func, '__name__', 'unknown')
        self.logs.append({
            'row': self.input_row,
            'col': self.output_column,
            'step': self.step,
            'role_type': role,
            'role': name,
            'status': status,
            'value': str(self.value),
            'details': details
        })

    def __or__(self, func):
        if self.stopped:
            self.step += 1; self._log(func, 'skipped'); return self
        self.step += 1
        role = getattr(func, '_role', 'validator')
        try:
            res = func(self.value)
            if role == 'validator':
                self.value = res; self.status = 'valid'; self._log(func, 'passed')
            elif role == 'transformer':
                self.value = res; self.status = 'success'; self.final_value = res; self.stopped = True; self._log(func, 'passed')
        except Exception as e:
            if role == 'validator':
                self.status = 'dirty'; self.final_value = None; self.stopped = True; self._log(func, 'failed', str(e))
            elif role == 'transformer':
                self._log(func, 'failed', str(e))
        return self

    def apply(self, pipeline):
        transformer_seen = False
        for func in pipeline:
            if getattr(func, '_role', 'validator') == 'transformer': transformer_seen = True
            self | func

        if self.status == 'success': pass
        elif self.status == 'dirty': self.final_value = None
        elif self.status in ['valid', 'pending']:
            if transformer_seen:
                self.status = 'dirty'; self.final_value = None
                self.logs.append({'row': self.input_row, 'col': self.output_column, 'step': self.step + 1, 'role_type': 'system', 'role': 'chain_exhausted', 'status': 'failed', 'value': str(self.value), 'details': 'All transformers failed'})
            else:
                self.status = 'success'; self.final_value = self.value
        return self

class SquishyEngine:
    def __init__(self, config_list, source_df):
        self.config = config_list; self.df = source_df; self.logs = []; self.final_df = None

    def run(self):
        print(f"Processing {len(self.df)} rows...")
        final_df = self.df.copy()
        all_logs = []

        for col_def in self.config:
            target = col_def['target']; source = col_def.get('source', target)
            pipeline = col_def['pipeline']; spec = TYPE_SPECS.get(col_def.get('type', 'string'))
            if source not in self.df.columns: continue

            def process(row):
                m = Monad(row[source], row.name, target).apply(pipeline)
                return (m.final_value if m.status == 'success' else spec['default'], m.logs)

            results = None
            if HAS_PARALLEL:
                try:
                    results = self.df.parallel_apply(process, axis=1)
                except Exception as e:
                    print(f"Parallel apply failed. Falling back to standard apply...")
                    results = None

            if results is None:
                results = self.df.apply(process, axis=1)

            final_df[target] = results.apply(lambda x: x[0])
            for log_list in results.apply(lambda x: x[1]): all_logs.extend(log_list)

            final_df[target] = spec['coerce'](final_df[target])

        self.final_df = final_df
        self.logs = pd.DataFrame(all_logs)
        return final_df

# --- 3. TOOLKIT ---
def validator(f): f._role = 'validator'; return f
def transformer(f): f._role = 'transformer'; return f

@validator
def must_exist(v):
    if pd.isna(v) or str(v).strip() == '' or str(v).lower() in ['n/a', 'null', 'nan', 'none', '<na>']: raise ValueError("Missing")
    return v

@transformer
def to_iso_code(v):
    m = {'TH': 'TH', 'USA': 'US', 'UK': 'UK', 'JP': 'JP', 'CN': 'CN'}
    res = m.get(str(v).strip().upper(), None)
    if res is None: raise ValueError("Mapping failed")
    return res

@transformer
def to_float(v): return float(str(v).replace(',', '').replace('$', '').replace('฿', ''))

@transformer
def parse_iso(v): return pd.to_datetime(v, format='%Y-%m-%d')
@transformer
def parse_us(v): return pd.to_datetime(v, format='%m/%d/%Y')
@transformer
def parse_uk(v): return pd.to_datetime(v, format='%d/%m/%Y')
@transformer
def parse_thai(v):
    s = str(v); parts = s.split('-')
    if len(parts) == 3 and parts[0].isdigit() and int(parts[0]) > 2400:
        return pd.to_datetime(f"{int(parts[0])-543}-{parts[1]}-{parts[2]}")
    raise ValueError("Not Thai Format")

# --- 4. DATA GENERATOR ---
def generate_complex_data(n=3000):
    print(f"Generating {n} rows of mock data...")
    dates = np.random.choice(['2024-01-01', '01/31/2024', '31/01/2024', '2567-01-01', 'NotDate'], n, p=[0.6, 0.1, 0.1, 0.1, 0.1])
    countries = np.random.choice(['TH', 'USA', 'UK', 'JP', 'BadCode', None], n, p=[0.6, 0.2, 0.1, 0.05, 0.025, 0.025])
    prices = np.random.choice(['100', '$50.00', '1,000', 'Free', None], n, p=[0.5, 0.3, 0.1, 0.05, 0.05])
    return pd.DataFrame({'date_col': dates, 'country_col': countries, 'price_col': prices})

# --- HELPER: Safely truncate massive strings for Parquet Metadata ---
def _truncate(val, max_len=40):
    s = str(val)
    return s if len(s) <= max_len else s[:max_len] + "..."

# --- 5. DE FEATURE: DATALAKE EXPORT ---
def export_to_datalake(engine, base_dir="datalake/", run_date=None):
    """Data Engineer Task: Export final data with embedded quality metadata in Parquet footer."""
    if engine.final_df is None:
        print("No data to save.")
        return

    os.makedirs(base_dir, exist_ok=True)

    current_run_date = run_date or datetime.datetime.now().strftime('%Y-%m-%d')

    export_data = engine.final_df.copy()
    export_data['run_date'] = current_run_date

    # Aggregate detailed logs into a lightweight JSON summary
    if not engine.logs.empty:
        summary_df = engine.logs.groupby(['col', 'step', 'role', 'role_type', 'status']).size().reset_index(name='count')

        failed_logs = engine.logs[engine.logs['status'] == 'failed']
        top_failed_list = []

        for keys, group in failed_logs.groupby(['col', 'step', 'role', 'role_type', 'status']):
            top_vals = group['value'].value_counts().head(5)
            top_str = ", ".join([f"'{_truncate(v)}' ({c})" for v, c in top_vals.items()])
            top_failed_list.append({
                'col': keys[0], 'step': keys[1], 'role': keys[2], 'role_type': keys[3], 'status': keys[4],
                'top_failed_values': top_str
            })

        if top_failed_list:
            top_failed_df = pd.DataFrame(top_failed_list)
            summary_df = summary_df.merge(top_failed_df, on=['col', 'step', 'role', 'role_type', 'status'], how='left')
        else:
            summary_df['top_failed_values'] = ""

        summary_df['top_failed_values'] = summary_df['top_failed_values'].fillna('')
        summary_df['run_date'] = current_run_date
        summary_json = summary_df.to_json(orient='records')
    else:
        summary_json = "[]"

    # Convert to PyArrow Table to inject metadata into the schema/footer
    table = pa.Table.from_pandas(export_data)

    custom_metadata = table.schema.metadata or {}
    custom_metadata[b'quality_summary'] = summary_json.encode('utf-8')
    table = table.replace_schema_metadata(custom_metadata)

    print(f"\n[DE] Saving Clean Data (with embedded Quality Summary) to Hive-Partitioned Parquet for {current_run_date}...")
    pq.write_to_dataset(
        table,
        root_path=os.path.join(base_dir, "clean_data"),
        partition_cols=['run_date'] # Modified to partition ONLY by date
    )
    print(f"[DE] Data successfully written to {base_dir}/clean_data")

# --- 6. QA FEATURE: VISUALIZATION DASHBOARD ---
def qa_load_and_visualize(datalake_dir="datalake/"):
    """QA Engineer Task: Extract metadata from Parquet footers and generate an interactive dashboard."""
    if alt is None:
        print("[QA] Cannot generate dashboard. Altair is missing.")
        return

    data_path = os.path.join(datalake_dir, "clean_data")
    if not os.path.exists(data_path):
        print(f"[QA] No data found in {data_path}")
        return

    print("\n[QA] Scanning Parquet file footers for embedded Quality Summaries...")
    all_summaries = []
    seen_summaries = set()

    for root, dirs, files in os.walk(data_path):
        for file in files:
            if file.endswith('.parquet'):
                file_path = os.path.join(root, file)
                schema = pq.read_schema(file_path)

                if schema.metadata and b'quality_summary' in schema.metadata:
                    summary_json = schema.metadata[b'quality_summary'].decode('utf-8')
                    if summary_json not in seen_summaries:
                        seen_summaries.add(summary_json)
                        all_summaries.extend(json.loads(summary_json))

    if not all_summaries:
        print("[QA] No quality summary metadata found in Parquet footers.")
        return

    print("[QA] Generating Interactive Vega-Lite Dashboard...")

    df_summary = pd.DataFrame(all_summaries).drop_duplicates()

    if 'top_failed_values' not in df_summary.columns:
        df_summary['top_failed_values'] = ""

    # Setup Interactive Partition Filter
    run_dates = [None] + list(df_summary['run_date'].dropna().unique())
    labels = ['All Partitions'] + list(df_summary['run_date'].dropna().unique())

    partition_dropdown = alt.binding_select(
        options=run_dates,
        labels=labels,
        name="Filter by Run Date: "
    )
    partition_selection = alt.selection_point(fields=['run_date'], bind=partition_dropdown)

    # CHART 1: Pipeline Operation Chain Flow
    chart_chain = alt.Chart(df_summary, title="Pipeline Operation Flow (Pass/Fail/Skip)").mark_bar().encode(
        x=alt.X('count:Q', stack='normalize'),
        y=alt.Y('role:N', sort=alt.EncodingSortField(field="step", order="ascending"), title="Pipeline Step"),
        color=alt.Color('status:N', scale=alt.Scale(domain=['passed', 'failed', 'skipped'], range=['#2ca02c', '#d62728', '#c7c7c7'])),
        row=alt.Row('col:N', header=alt.Header(titleOrient="top", labelOrient="top")),
        tooltip=['col', 'role', 'status', 'count', alt.Tooltip('top_failed_values:N', title='Top Failed Values')]
    ).add_params(
        partition_selection
    ).transform_filter(
        partition_selection
    ).properties(width=600, height=100).resolve_scale(y='independent')

    # CHART 2: AI Recommender (Efficiency Scores)
    df_stats = df_summary[df_summary['role_type'] == 'transformer'].copy() if 'role_type' in df_summary.columns else pd.DataFrame()

    chart_rec = alt.Chart(df_stats, title="AI Recommender (Transformer Pass Volume)").mark_bar().encode(
        x=alt.X('sum(count):Q', title="Total Passed"),
        y=alt.Y('role:N', sort='-x'),
        color=alt.Color('col:N'),
        tooltip=['col', 'role', 'sum(count)']
    ).transform_filter(
        alt.datum.status == 'passed'
    ).add_params(
        partition_selection
    ).transform_filter(
        partition_selection
    ).properties(width=600, height=150)

    final_dashboard = alt.vconcat(chart_chain, chart_rec).resolve_scale(color='independent')

    if in_notebook():
        from IPython.display import display
        display(final_dashboard)
    else:
        out_file = 'qa_dashboard.html'
        final_dashboard.save(out_file)
        print(f"[QA] Dashboard exported locally to {out_file} (Open in your browser)")

# --- 7. MAIN ORCHESTRATION ---
def main():
    config = [
        {"target": "clean_date", "source": "date_col", "type": "datetime", "pipeline": [must_exist, parse_thai, parse_uk, parse_us, parse_iso]},
        {"target": "country_code", "source": "country_col", "type": "string", "pipeline": [must_exist, to_iso_code]},
        {"target": "price", "source": "price_col", "type": "float", "pipeline": [must_exist, to_float]}
    ]

    # Simulate 5 days of pipeline execution
    base_date = datetime.datetime.now()

    for i in range(5):
        run_date = (base_date - datetime.timedelta(days=i)).strftime('%Y-%m-%d')
        print(f"\n{'='*50}")
        print(f"=== Simulating Pipeline Run for {run_date} ===")
        print(f"{'='*50}")

        # Generating 600 rows per day to total 3000 over 5 days
        df = generate_complex_data(600)

        engine = SquishyEngine(config, df)
        engine.run()

        export_to_datalake(engine, base_dir="my_datalake/", run_date=run_date)

    print(f"\n{'='*50}")
    print(f"=== Generating QA Dashboard ===")
    print(f"{'='*50}")
    qa_load_and_visualize(datalake_dir="my_datalake/")

if __name__ == "__main__":
    main()


=== Simulating Pipeline Run for 2026-06-25 ===
Generating 600 rows of mock data...
Processing 600 rows...

[DE] Saving Clean Data (with embedded Quality Summary) to Hive-Partitioned Parquet for 2026-06-25...
[DE] Data successfully written to my_datalake//clean_data

=== Simulating Pipeline Run for 2026-06-24 ===
Generating 600 rows of mock data...
Processing 600 rows...

[DE] Saving Clean Data (with embedded Quality Summary) to Hive-Partitioned Parquet for 2026-06-24...
[DE] Data successfully written to my_datalake//clean_data

=== Simulating Pipeline Run for 2026-06-23 ===
Generating 600 rows of mock data...
Processing 600 rows...

[DE] Saving Clean Data (with embedded Quality Summary) to Hive-Partitioned Parquet for 2026-06-23...
[DE] Data successfully written to my_datalake//clean_data

=== Simulating Pipeline Run for 2026-06-22 ===
Generating 600 rows of mock data...
Processing 600 rows...

[DE] Saving Clean Data (with embedded Quality Summary) to Hive-Partitioned Parquet for 2026

alt.VConcatChart(...)